# Cells

**What you'll learn.** The numbered tutorials introduce one idea at a time on a
handful of entities. This notebook is the single-cell *deep dive*: every cell
model embpy can reach, run on the same cells, then scored three different ways.

The two scoring sections answer two different questions, and the difference
matters more than any individual number. There is no single "which model is
best" ranking, and a notebook that produced one would be lying to you.

| Section | Question | What it needs from the data |
| --- | --- | --- |
| [1. scIB](#1-scib-does-integration-preserve-the-biology) | does integration remove the batch and keep the biology? | cell-type labels plus a technical covariate |
| [2. cell-eval](#2-cell-eval-is-the-perturbation-preserved) | does the embedding preserve a perturbation effect? | many perturbation levels |

Because those two needs conflict, this notebook uses **two** datasets, both from
`pertpy`. Section 2 runs on a different object than section 1, and
[the dataset section](#the-datasets-and-the-contract-you-can-swap) says why.

**Evaluation here is scIB and cell-eval, and nothing else.** embpy ships its own
comparison metrics too, and this notebook deliberately does not use them. For
cells there are community-standard answers to both questions, and putting a
parallel bespoke suite beside them invites averaging numbers that are not on the
same scale. Those tools earn their place in [genes](genes.ipynb),
[proteins](proteins.ipynb) and [small molecules](small_molecules.ipynb), where no
standard exists.

**Prerequisites.** [04_benchmark_models](04_benchmark_models.ipynb) is the short
version of both sections -- five models, one dataset, no batch covariate.
This notebook is the long version, and the difference is not length: nb04 had no
batch variable at all, so it could not ask the integration question, and it used
cell type as a stand-in perturbation, which made every discrimination score come
back exactly 1.000. Both of those are fixed here, and both are shown rather than
asserted.

## Requirements

The single-cell backends do not coexist with embpy's other extras: `helical`
pins `numpy<2.3` and `transformers<=4.51.3`, `arc-state` wants
`transformers>=4.52.3`, and `cell-eval` wants `numpy>=2.4.2`. Those three
constraints are mutually unsatisfiable, so a single `uv pip install embpy[all]`
cannot work and no amount of retrying will make it.

The environment this notebook was executed in resolves that by installing
`helical` with `--no-deps` and adding its real runtime requirements by hand:

```bash
uv venv --python 3.13 .venv-sc
uv pip install --python .venv-sc/bin/python -e .
uv pip install --python .venv-sc/bin/python \
    scanpy scib scib-metrics cell-eval pdex scvi-tools
uv pip install --python .venv-sc/bin/python --no-deps helical
uv pip install --python .venv-sc/bin/python arc-state arc-stack
uv pip install --python .venv-sc/bin/python ipykernel
.venv-sc/bin/python -m ipykernel install --user \
    --name embpy-sc --display-name "Python (embpy-sc)"
```

Then select the `Python (embpy-sc)` kernel. Two practical points:

* **`.venv-sc` has no `pip`.** It was created by uv, so `python -m pip install`
  fails with `No module named pip`. Use `uv pip install --python
  .venv-sc/bin/python ...` for everything.
* **`pertpy` is not installed there, deliberately.** It is not an embpy
  dependency, and adding it to this environment risks moving `numpy` or
  `scanpy` in a resolution that took real work to get right. The loaders below
  read a staged `.h5ad` when one exists and only call `pertpy` as a fallback, so
  you stage the two files once from any environment that has pertpy.

> **What is installed is metrics, not methods.** `scib` and `scib-metrics` give
> you the scIB *scores*. `harmonypy`, `scanorama` and `bbknn` are not installed,
> so there is no explicit batch-correction step here to compare against. What
> section 2 measures is whatever integration each embedding model performs
> *implicitly*, which is a narrower claim than "we benchmarked integration
> methods" and worth keeping straight.

In [ ]:
import os
import time
import warnings
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from IPython.display import display

import embpy
from embpy import BioEmbedder, pl, pp, tl

# The single-cell stack is chatty: scib warns about the leidenalg backend on
# every resolution it tries, and scanpy warns when a copy densifies. Neither
# changes a result, and 40 repeats of each would bury the tables.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 1

OUTPUT_DIR = Path("outputs")
DATA_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

embedder = BioEmbedder(device="auto", organism="human")
print(f"embpy {embpy.__version__}")

## The datasets, and the contract you can swap

Two objects, because the three scoring sections need differently shaped
experiments and no single dataset provides all of it:

| | `adata` -- an **atlas** | `pert_adata` -- `norman_2019` |
| --- | --- | --- |
| used by | the sweep, section 1, attention, decode | section 2 only |
| needs | cell-type labels plus a **technical** covariate | many perturbation levels plus a control |
| the covariate is | donor, lab, sequencing technology, 10x chemistry | -- |

**Section 1 needs an atlas, not a perturbation experiment.** scIB asks whether
merging batches removes a *technical* axis while preserving the *biological*
one. Point it at a treatment experiment and a model scores well for making
stimulated cells look like control cells -- which is not integration, it is
deleting the result. So the atlas is what section 1 runs on, and section 2
deliberately uses a different object.

The mirror of that rule applies to section 2. cell-eval scores agreement *per
perturbation*, so an atlas gives it nothing to work with, and a dataset with a
single treatment arm gives it almost nothing.
[04_benchmark_models](04_benchmark_models.ipynb) is the worked failure: with 7
stand-in perturbations it returned a discrimination score of exactly 1.000 for
every model, ranking nothing at all.

**Everything after this section reads five constants and nothing else about the
data.** To use your own atlas, repoint `ATLAS_PATH` or `load_atlas`, then check
it against the requirements this section asserts.

In [ ]:
# ------------------------------------------------------------ THE CONTRACT
# Repoint these to swap in your own data. Nothing downstream hardcodes a
# column name, so a rename here propagates through every later section.
LABEL_KEY = "cell_type"      # ground-truth biology; scIB bio-conservation
BATCH_KEY = "batch"          # TECHNICAL covariate; scIB batch-correction
COUNTS_LAYER = "counts"      # raw integer counts, kept across preprocessing

N_CELLS = 3000               # subsample target, to keep a run to minutes
SEED = 0
MIN_CELLS_PER_LABEL = 4      # below this, per-label statistics are meaningless
MIN_PER_LABEL = 60           # floor per label, so rare types survive sampling

# The atlas: the scIB human pancreas benchmark (Luecken et al. 2022). 16,382
# cells over 19,093 genes, 14 islet cell types, and nine batches spanning six
# protocols -- four inDrop runs plus CEL-Seq, CEL-Seq2, Fluidigm C1, SMARTer and
# Smart-seq2. It is the canonical integration benchmark because the technical
# axis is genuinely large: median library size runs from ~5,000 UMIs in the
# droplet runs to ~1.3 million reads in Fluidigm C1.
#
# To swap in your own atlas, repoint ATLAS_PATH and check it against the
# assertions in the next cell. Anything with cell-type labels, a technical
# covariate that is not nested inside them, counts, and unique symbols works.
ATLAS_URL = "https://ndownloader.figshare.com/files/46763269"
ATLAS_PATH = DATA_DIR / "scIBPancreas.h5ad"


def load_atlas():
    # Staging is the intended path: it keeps this kernel free of dataset
    # dependencies and makes the choice of atlas explicit rather than a silent
    # default buried in a helper.
    if not ATLAS_PATH.exists():
        import urllib.request  # noqa: PLC0415 - one-off fetch, for whoever stages it

        print(f"fetching the atlas to {ATLAS_PATH} (316 MB, once) ...")
        urllib.request.urlretrieve(ATLAS_URL, ATLAS_PATH)
    return sc.read_h5ad(ATLAS_PATH)


adata = load_atlas()
print(f"atlas: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"obs columns: {list(adata.obs.columns)}")

### Assert the contract, do not trust it

Six things every later section depends on, checked here rather than discovered
as a confusing failure eight sections later.

Raw integer counts are the strictest of them. `scvi` and `scanvi` declare
`input_layer="counts"`, and Geneformer tokenises a cell by gene *rank*, so a
pre-normalised matrix does not raise -- it quietly produces a different, wrong
answer. Atlases are often published already normalised, which is exactly the
case this assertion catches.

In [ ]:
# Resolve the contract columns against whatever this atlas calls them, so a
# dataset using "donor" or "tech" needs one edit here and none below.
COLUMN_ALIASES = {
    LABEL_KEY: ["cell_type", "cell_type_annotation", "celltype", "labels", "louvain"],
    BATCH_KEY: ["batch", "donor", "cell_source", "tech", "study", "sample"],
}
for canonical, candidates in COLUMN_ALIASES.items():
    if canonical in adata.obs.columns:
        continue
    found = next((c for c in candidates if c in adata.obs.columns), None)
    if found is None:
        raise KeyError(
            f"no column for {canonical!r}; tried {candidates}. "
            f"Available: {list(adata.obs.columns)}"
        )
    adata.obs[canonical] = adata.obs[found]
    print(f"mapped {canonical!r} <- {found!r}")

# Subsample *within* label, with a floor. A flat random sample of 1,500 from
# 16,382 cells leaves schwann and t_cell with one cell each -- and the rare,
# batch-restricted labels are precisely what isolated_label_asw exists to
# score, so deleting them would remove the metric's subject matter.
rng = np.random.default_rng(SEED)
labels = adata.obs[LABEL_KEY].astype(str)
quota = max(MIN_PER_LABEL, N_CELLS // labels.nunique())

keep = []
for level in labels.unique():
    idx = np.flatnonzero((labels == level).to_numpy())
    if len(idx) > quota:
        idx = rng.choice(idx, quota, replace=False)
    keep.append(idx)
adata = adata[np.sort(np.concatenate(keep))].copy()

for key in (LABEL_KEY, BATCH_KEY):
    adata.obs[key] = adata.obs[key].astype(str).astype("category")

# Prefer an explicit counts layer over .X. Published atlases frequently ship
# normalised values in .X and keep the counts beside them, and this one does:
# .X here is log-normalised (max ~13), while .layers["counts"] holds the counts.
if COUNTS_LAYER in adata.layers:
    adata.X = adata.layers[COUNTS_LAYER].copy()
else:
    adata.layers[COUNTS_LAYER] = adata.X.copy()


def nonzero_values(matrix):
    # A matrix here may be sparse or dense depending on how the atlas was
    # written, and `.data` means different things for each -- on a dense array
    # it is the raw memory buffer, not the values, so reading it silently
    # produces nonsense rather than an error.
    if sparse.issparse(matrix):
        return matrix.data
    flat = np.asarray(matrix).ravel()
    return flat[flat != 0]


def round_matrix(matrix):
    if sparse.issparse(matrix):
        out = matrix.copy()
        out.data = np.round(out.data)
        return out
    return np.round(np.asarray(matrix))


X = adata.layers[COUNTS_LAYER]

# Not every value in this atlas's count layer is an integer, and the reason is
# bookkeeping rather than corruption: the droplet runs contribute integer UMIs,
# while the plate-based studies (Smart-seq2, SMARTer, Fluidigm C1) contribute
# *estimated* counts from transcript quantification, which are fractional by
# construction -- they sit at 1.002, 2.008, 4.032. scvi-tools requires integers
# and refuses the fractional ones, so round. That moves those values by well
# under a percent.
vals = nonzero_values(X)
frac_integral = float(np.mean(np.abs(vals - np.round(vals)) < 1e-6))
if frac_integral < 1.0:
    print(f"{1 - frac_integral:.1%} of nonzero count values are fractional "
          f"(estimated counts from the plate-based protocols); rounding")
    adata.layers[COUNTS_LAYER] = round_matrix(X)
    adata.X = round_matrix(X)
    X = adata.layers[COUNTS_LAYER]

vals = nonzero_values(X)
integral = float(np.abs(vals - np.round(vals)).max()) == 0.0

assert integral, (
    "X must hold raw integer counts. scvi/scanvi declare input_layer='counts' "
    "and Geneformer tokenises by rank; a normalised matrix fails silently."
)
assert adata.obs[LABEL_KEY].nunique() >= 4, "scIB bio-conservation needs >= 4 labels"
assert adata.obs[BATCH_KEY].nunique() >= 2, "scIB batch metrics need >= 2 batches"
assert adata.var_names.is_unique, "gene symbols must be unique"
assert adata.n_vars >= 2000, (
    f"only {adata.n_vars} genes; the rank tokenisers (Geneformer, scGPT, UCE) "
    "need thousands before their embeddings mean anything"
)

n_nonzero = X.nnz if sparse.issparse(X) else int(np.count_nonzero(X))
print(f"counts integral, {1 - n_nonzero / (X.shape[0] * X.shape[1]):.1%} zeros, "
      f"X.max() = {float(X.max()):.0f}")
print(f"{LABEL_KEY}: {adata.obs[LABEL_KEY].nunique()} levels | "
      f"{BATCH_KEY}: {adata.obs[BATCH_KEY].nunique()} levels")
print(f"var_names: {list(adata.var_names[:4])}")

The label-by-batch cross-tabulation decides whether section 1 can mean anything,
so read it rather than skipping it. If every cell type appears in every batch,
then "mix the batches" and "keep the cell types apart" are separable requests.
If a cell type lives in only one batch they are in direct conflict, no model can
satisfy both, and a low score would be measuring the experimental design rather
than the model.

In [ ]:
crosstab = pd.crosstab(adata.obs[LABEL_KEY], adata.obs[BATCH_KEY])
crosstab["total"] = crosstab.sum(axis=1)
display(crosstab.sort_values("total", ascending=False))

tiny = crosstab.index[crosstab["total"] < MIN_CELLS_PER_LABEL].tolist()
print(f"labels below {MIN_CELLS_PER_LABEL} cells: {tiny or 'none'}")

# How nested is the design? 0 means every label is spread evenly across
# batches; the printed maximum means each label sits in exactly one batch.
n_batches = adata.obs[BATCH_KEY].nunique()
shares = crosstab.drop(columns="total").div(crosstab["total"], axis=0)
NESTEDNESS = float((shares - 1.0 / n_batches).abs().sum(axis=1).mean())
print(f"nestedness {NESTEDNESS:.3f}  "
      f"(0 = balanced, {2 * (1 - 1 / n_batches):.2f} = fully nested)")

### Why a real dataset, and not a simulation

The obvious way to write this notebook is to simulate cells: a few hundred rows,
a Poisson count matrix, a handful of marker genes. The notebook this one
replaces did exactly that over 24 genes, and every number in it was noise.

The reason is structural rather than a matter of scale. scGPT, Geneformer, UCE,
STATE and STACK all tokenise a cell by the *rank order of thousands of genes*.
Give them 24 genes and every cell yields nearly the same token sequence, so
every cell gets nearly the same embedding -- and comparing near-constant
embeddings compares rounding error. That is what the `n_vars >= 2000` assertion
above guards.

Fabricating the *covariates* does not work either. Three designs were built and
measured on `pbmc3k` before this notebook committed to real data. Silhouette on
PCA(30), 600 cells:

| Design | biology | injected covariate | verdict |
| --- | --- | --- | --- |
| `pbmc3k`, untouched | +0.101 | -- | no batch or perturbation exists to measure |
| injected batch, sigma = 0.5 | +0.064 | +0.034 | the covariate barely registers |
| injected batch, sigma = 1.5 | **-0.010** | +0.118 | visible only by destroying the biology |
| injected perturbation, 8x on 18 genes | +0.101 | -0.005 | invisible at every strength tried |
| library-depth tertile as batch | +0.101 | -0.067 | and 0.714 confounded with cell type -- it *is* cell type |

Two reasons those failed, both of which generalise past this notebook:

* **`normalize_total` removes a library-size shift by construction.** Half of
  the fabricated batch effect was a per-cell scale factor, and normalising is
  the first thing every one of these models wants. Injecting signal into a
  quantity the pipeline exists to divide out cannot survive the pipeline.
* **An effect on 18 genes out of 32,738 does not survive HVG selection.** Real
  signatures are hundreds of genes wide *and* cell-type dependent. A fixed short
  list scaled by a constant is neither, so PCA never sees it.

### Is this atlas a real integration benchmark?

Section 1 scores every embedding on how well it removes `BATCH_KEY`. That score
is only interesting if there is a batch effect to remove, so measure it here,
before any model runs, and let the measurement set expectations.

Measure it two ways, because the two answer different questions and only one of
them is the right basis for a verdict.

**Globally**, a batch is almost never a coherent cluster: cells group by cell
type first, so the global batch silhouette sits near zero *however strong the
batch effect is*. Judging a dataset on that number would call a badly batched
atlas clean.

**Within a cell type** is the question that matters -- holding biology fixed, can
you still tell the protocols apart? If yes, there is a technical axis for
integration to remove. If no, the covariate is either absent or so entangled
with biology that removing it would mean merging cell types.

So the verdict below uses the median within-label figure, and reports the global
one beside it precisely so the gap between them is visible.

For this atlas the technical axis is not subtle. Median library size runs from
about 5,000 UMIs in the droplet runs to 1.3 million reads in Fluidigm C1 -- a
250-fold spread that has nothing to do with pancreatic biology, and everything
to do with which protocol ran.

In [ ]:
from sklearn.metrics import silhouette_score

# A throwaway PCA purely to size the problem. This is not one of the
# embeddings under test; it exists so the prediction below is measured
# rather than assumed.
probe = adata.copy()
sc.pp.normalize_total(probe, target_sum=1e4)
sc.pp.log1p(probe)
sc.pp.highly_variable_genes(probe, n_top_genes=2000)
probe = probe[:, probe.var.highly_variable].copy()
sc.pp.scale(probe, max_value=10)
sc.tl.pca(probe, n_comps=30)
E = probe.obsm["X_pca"]

big_labels = [
    lab for lab, n in probe.obs[LABEL_KEY].value_counts().items() if n >= 40
][:4]

rows = [{
    "covariate": f"{LABEL_KEY} (the biology to keep)",
    "global": silhouette_score(E, probe.obs[LABEL_KEY].astype(str)),
}]
batch_row = {
    "covariate": f"{BATCH_KEY} (the axis to remove)",
    "global": silhouette_score(E, probe.obs[BATCH_KEY].astype(str)),
}
for lab in big_labels:
    mask = (probe.obs[LABEL_KEY] == lab).to_numpy()
    batch_row[f"within {lab}"] = (
        silhouette_score(E[mask], probe.obs[BATCH_KEY][mask].astype(str))
        if probe.obs[BATCH_KEY][mask].nunique() > 1 else np.nan
    )
rows.append(batch_row)

COVARIATE_STRENGTH = pd.DataFrame(rows).set_index("covariate")
display(COVARIATE_STRENGTH.round(3))

# The verdict keys off the WITHIN-LABEL numbers, not the global one, and the
# reason is not a detail. Globally, cells group by cell type -- biology
# dominates -- so a batch is never a coherent global cluster and its silhouette
# sits near zero however strong the batch effect is. Judging the dataset on the
# global figure would call a strongly batched atlas "clean". The within-label
# figure asks the question that matters: holding cell type fixed, can you still
# tell the protocols apart?
within_cols = [c for c in COVARIATE_STRENGTH.columns if c.startswith("within ")]
within = COVARIATE_STRENGTH.loc[f"{BATCH_KEY} (the axis to remove)", within_cols]
batch_within = float(np.nanmedian(within.astype(float).values))
batch_global = float(batch_row["global"])

WEAK_BATCH = batch_within < 0.05
print(f"\nbatch silhouette: global {batch_global:+.3f}, "
      f"median within-label {batch_within:+.3f}")
print(
    "WEAK integration benchmark: expect batch_correction high and near-tied "
    "for every model, ranking nothing."
    if WEAK_BATCH else
    "STRONG integration benchmark: a real technical axis survives inside every "
    "cell type, so expect batch_correction to separate the models."
)
if batch_global < 0.05 <= batch_within:
    print("Note the disagreement: the global figure is near zero because cell "
          "type dominates the global geometry, not because the batch effect is "
          "small. This is why the verdict uses the within-label median.")
del probe, E

## Which cell models does embpy have?

Twenty-five entries live in the single-cell registry
(`src/embpy/models/singlecell_models.py`), and they are not twenty-five
interchangeable choices. They differ in four ways that decide whether a model
will work on *your* AnnData at all:

* **`vocab_type`** -- whether the model wants gene symbols, Ensembl IDs, or does
  not care. Get this wrong on a `symbol` model and you get zero matches and an
  empty embedding, with no error. [Part 2](#preprocess-once-per-requirement-not-once-per-model)
  measures the overlap before embedding for exactly that reason.
* **`input_layer`** -- raw counts, `.layers["counts"]`, or log-normalised
  expression. `preprocessing="auto"` reads this and prepares the right one.
* **`supports_decode`** -- whether the latent space can be projected back to
  expression. Five keys can; the rest are encoders only.
* **`supports_generation`** -- whether whole profiles can be synthesised. Exactly
  one key can.

In [ ]:
from embpy.models.singlecell_models import list_singlecell_models, singlecell_info

# The models this notebook attempts. The first five are the ones
# 04_benchmark_models.ipynb proved end to end; the next three have weights
# cached on this cluster and are attempted with failures reported, not raised.
ROSTER = [
    "pca", "scvi", "scgpt", "geneformer_v2_12L", "state",
    "uce", "transcriptformer_sapiens", "tahoe_70m",
]
# Not new models -- the same scvi-tools wrapper, told different things. These
# two exist to make section 2's comparison honest; see the tiers table there.
INTEGRATION_VARIANTS = ["scvi_batch", "scanvi"]
SKIPPED = ["stack", "cell2sentence_2b", "tahoe_1b", "tahoe_3b", "totalvi"]

rows = []
for key in ROSTER + SKIPPED:
    card = singlecell_info(key)
    rows.append({
        "model": key,
        "wrapper": card.wrapper_class_name,
        "dim": card.embedding_dim,
        "vocab": card.vocab_type,
        "input_layer": card.input_layer,
        "auto_prep": card.default_preprocessing,
        "decode": card.supports_decode,
        "generate": card.supports_generation,
        "attempted": key in ROSTER,
    })

CATALOGUE = pd.DataFrame(rows).set_index("model")
display(CATALOGUE)

print(f"registry holds {len(list_singlecell_models())} single-cell models; "
      f"this notebook attempts {len(ROSTER)}")

**Skipped on purpose.** A catalogue that quietly omits what it could not run is
not a catalogue, so here is each exclusion with its reason:

* **`stack`** -- the only `supports_generation=True` entry, and its checkpoint
  *is* on disk. It imports cleanly and then fails inside arc-stack's own h5ad
  reader, upstream of anything embpy controls. Its absence is why this notebook
  has no generation section, which is a real gap rather than an oversight.
* **`cell2sentence_2b` / `_27b`** -- 4.9 GB of Gemma-2 weights, and the LLM
  tokenisation turns a cell into a gene *sentence*, so an attention tensor over
  it is enormous.
* **`tahoe_1b`, `tahoe_3b`** -- only the 70M checkpoint is cached; these would
  download.
* **`totalvi`** -- needs paired protein counts, which this dataset does not have.
* **the other eight Geneformer variants** -- v1 at 6L and 12L, the CZI
  fine-tune, v2 at 20L and 18L, and three cancer/104M variants. Only
  `v2/gf-12L-38M-i4096` is cached here, so `geneformer_v2_12L` is the one that
  runs without a download. The other eight are a real catalogue rather than
  padding: swapping to a 316M-parameter variant is a one-string change and a
  download.

The registry keys are worth cross-checking against what the embedder will
actually accept, because a key in the registry that the embedder cannot resolve
is a catalogue entry you cannot use.

In [ ]:
available = set(embedder.list_available_models("single_cell"))
registry = set(list_singlecell_models())

print(f"registry: {len(registry)} | embedder reports available: {len(available)}")
only_registry = sorted(registry - available)
only_available = sorted(available - registry)
print(f"in registry but not reported available: {only_registry or 'none'}")
print(f"reported available but not in registry: {only_available or 'none'}")

missing = [k for k in ROSTER if k not in available]
if missing:
    print(f"\nWARNING: roster entries the embedder cannot resolve: {missing}")
else:
    print(f"\nall {len(ROSTER)} roster entries resolve")

## Preprocess once per requirement, not once per model

There is no single "preprocess the data" step here, because the models do not
want the same input. The registry records what each one eats, and the three
answers are incompatible:

* `pca` wants **log-normalised** expression, from `.layers["log_normalized"]`.
* `scvi`, `scanvi` and `totalvi` want **raw counts**, from `.layers["counts"]`.
* the seven transformers want **raw counts in `.X`**, which they then tokenise
  themselves.

`preprocessing="auto"` reads those cards and prepares whatever the requested
models need. `resolve_singlecell_preprocessing` is the function that decides,
and it is worth calling directly once so the decision is visible rather than
implicit.

The rule it applies is deliberately conservative: if *any* requested model needs
processed expression, the whole call is lifted to `"standard"`. That is safe in
one direction only, and the reason is worth stating -- the standard pipeline
*preserves* raw counts in `.X` and `.layers["counts"]`, so a raw-count model can
still find what it needs afterwards. The converse is not true, which is why the
lift goes this way and not the other.

In [ ]:
from embpy.models.singlecell_models import (
    resolve_singlecell_preprocessing,
    singlecell_info,
)

resolved, report = resolve_singlecell_preprocessing(ROSTER, requested="auto")
print(f"resolved preprocessing for the whole roster: {resolved!r}")
print(f"reason: {report['reason']}\n")

REQUIREMENTS = pd.DataFrame(report["requirements"]).T
REQUIREMENTS.index.name = "model"
display(REQUIREMENTS)

Read the `input_layer` column against `default_preprocessing`. Those two are not
the same question: `default_preprocessing` is what the pipeline has to *produce*,
`input_layer` is what the wrapper then *reads*. `pca` is the only model here that
moves both away from raw, and it is the only one with `uses_hvg` set, which is
why one classical baseline drags the whole call to `"standard"`.

**Why this matters more than it looks.** Feed every model one log-normalised
matrix and nothing raises. Geneformer will happily rank log-normalised values,
scVI will fit a negative-binomial likelihood to non-integers, and both return
confident embeddings that are answering a different question than you asked.
Silent wrongness is the failure mode this section exists to prevent, which is
why the assertion in [part 1](#assert-the-contract-do-not-trust-it) checks
integrality rather than trusting the loader.

In [ ]:
# Preprocess once, into a copy, so `adata` keeps the untouched counts and every
# model can be given whichever representation its card asks for.
t0 = time.perf_counter()
prepared = pp.preprocess_counts(
    adata,
    pipeline=resolved,
    min_genes=0,       # QC already applied upstream; filtering here would
    min_cells=0,       # silently change n_obs and break the contract asserts
    target_sum=1e4,
    log_transform=True,
    n_top_genes=2000,
    select_hvg=True,
    scale=False,       # scaling is a PCA convenience, not a model requirement
    copy=True,
)
print(f"preprocess_counts({resolved!r}) took {time.perf_counter() - t0:.1f}s")
print(f"layers: {list(prepared.layers.keys())}")
print(f"var columns added: {[c for c in prepared.var.columns if c not in adata.var.columns]}")
print(f"highly variable genes: {int(prepared.var['highly_variable'].sum())}")

# The load-bearing check: raw counts must survive the standard pipeline,
# because five of the roster's models read them afterwards.
raw_after = prepared.layers[COUNTS_LAYER]
print(f"\n{COUNTS_LAYER!r} still integral after preprocessing: "
      f"{float(np.abs(raw_after.data - np.round(raw_after.data)).max()) == 0.0}")
print(f"log_normalized present: {'log_normalized' in prepared.layers}")

## The vocabulary, and the silent failure it causes

Every transformer in the roster treats a cell as a sentence whose words are
genes. That sentence is drawn from a **vocabulary**: a fixed inventory of gene
tokens, decided at pre-training and frozen. A gene the model has no token for
cannot appear in the sentence at all.

Two separate things travel under that name, and they fail differently.

**The identifier convention.** Some models were trained on gene symbols
(`TP53`), others on Ensembl IDs (`ENSG00000141510`). The registry records this as
`vocab_type`:

| `vocab_type` | Models | Meaning |
| --- | --- | --- |
| `symbol` | `scgpt`, `uce`, `state`, `stack` | wants symbols; Ensembl IDs match nothing |
| `either` | `geneformer_*`, `transcriptformer_*`, `tahoe_*` | the wrapper maps internally |
| `any` | `pca`, `scvi`, `scanvi`, `totalvi` | identifier-agnostic; it never looks up a gene |

The registry docstring states the consequence of getting it wrong plainly:
Ensembl IDs handed to a `symbol` model *"produce zero matches and the model
silently returns empty embeddings"*. No exception, no warning from the model
itself -- just a matrix of nothing that flows into every downstream table.

**The gene set.** Even with the right convention, your genes may simply not be
in the model's inventory: a targeted panel, another species, or symbol aliases
that have since been renamed.

In [ ]:
# embpy detects the convention by sampling var_names and matching the Ensembl
# pattern: >= 80% means ensembl_id, <= 20% means symbol, in between is "mixed"
# (which it warns about and leaves alone -- a mixed index cannot be converted
# safely in either direction).
detected = BioEmbedder._detect_vocab_type(adata.var_names)
print(f"detected convention of adata.var_names: {detected!r}")
print(f"sample: {list(adata.var_names[:3])}\n")

rows = []
for key in ROSTER:
    card = singlecell_info(key)
    # _ensure_singlecell_vocabulary reports what it *would* do without
    # committing to it, which is what makes it usable as an audit.
    _, plan = embedder._ensure_singlecell_vocabulary(adata, key, organism="human")
    rows.append({
        "model": key,
        "vocab_type": card.vocab_type,
        "action": plan["action"],
        "reason": plan.get("reason", ""),
    })

VOCAB_PLAN = pd.DataFrame(rows).set_index("model")
display(VOCAB_PLAN)

### Why the *size* of the overlap matters, not just its sign

A reader might reasonably assume that out-of-vocabulary genes are simply
dropped, costing you signal in proportion to how many were lost. For Geneformer
that is not what happens, and the real behaviour is worse and more interesting.

Geneformer ranks each cell's genes by expression divided by that gene's median
across its entire pre-training corpus, then takes the top 4096 as the token
sequence. An out-of-vocabulary gene therefore never competes for a rank slot.
Remove genes and you change *which* genes make the cut -- so the overlap changes
the representation of the genes that were in-vocabulary all along. It is not a
proportional loss of signal; it is a change of input.

That is the argument for measuring the overlap up front rather than reasoning
about it afterwards. The vocabularies ship as files next to the weights, so this
is a lookup rather than an inference.

In [ ]:
import glob
import json
import os
import pickle

HELICAL_CACHE = os.path.expanduser("~/.cache/helical/models")


def load_model_vocabulary(model_key):
    # Returns (set_of_genes, keyed_on, source_path) or (None, None, reason).
    # Vocabularies live beside the weights in formats that differ per model, so
    # this reads the two that ship a plain lookup table and reports honestly
    # for the rest rather than guessing.
    if model_key == "scgpt":
        hits = glob.glob(f"{HELICAL_CACHE}/scgpt/**/vocab.json", recursive=True)
        if not hits:
            return None, None, "vocab.json not in the helical cache"
        with open(hits[0]) as fh:
            vocab = json.load(fh)
        genes = {g for g in vocab if not g.startswith("<")}
        return genes, "symbol", hits[0]
    if model_key.startswith("geneformer"):
        hits = glob.glob(
            f"{HELICAL_CACHE}/geneformer/**/token_dictionary*.pkl", recursive=True
        )
        if not hits:
            return None, None, "token_dictionary.pkl not in the helical cache"
        with open(hits[0], "rb") as fh:
            tokens = pickle.load(fh)
        genes = {g for g in tokens if not str(g).startswith("<")}
        return genes, "ensembl_id", hits[0]
    return None, None, "no plain lookup table ships with this model"


rows = []
for key in ROSTER:
    genes, keyed_on, source = load_model_vocabulary(key)
    if genes is None:
        rows.append({"model": key, "vocab_size": np.nan, "keyed_on": "-",
                     "recognised": np.nan, "fraction": np.nan, "note": source})
        continue
    # Compare on the convention the vocabulary is keyed on, not ours.
    ours = set(adata.var_names)
    if keyed_on == "ensembl_id" and detected == "symbol":
        note = "our symbols vs an Ensembl vocabulary -- the wrapper maps these"
        overlap = np.nan
    else:
        overlap = len(ours & genes)
        note = ""
    rows.append({
        "model": key,
        "vocab_size": len(genes),
        "keyed_on": keyed_on,
        "recognised": overlap,
        "fraction": overlap / adata.n_vars if overlap is not np.nan else np.nan,
        "note": note,
    })

VOCAB_OVERLAP = pd.DataFrame(rows).set_index("model")
display(VOCAB_OVERLAP)

Two things to take from that table.

**The vocabularies are not the same size, and not by a little.** scGPT carries
roughly three times as many gene tokens as Geneformer, and the extra entries are
largely clone-named lncRNAs (`RP5-973N23.5`, `AC008079.12`) rather than
protein-coding genes. Two models trained on the same species differ by tens of
thousands of genes in what they can even represent. Neither choice is wrong, but
they are answering questions about different transcriptomes.

**A blank `recognised` is not a failure.** Where a vocabulary is keyed on
Ensembl IDs and this AnnData carries symbols, comparing the two sets directly
would report a spurious zero. Those rows are the ones `vocab_type="either"`
covers, where the wrapper does the mapping internally with its own alias
dictionary -- 173,697 entries in Geneformer's case. Reporting NaN and saying why
is the honest answer; reporting 0 would be a bug dressed as a finding.

> **The models whose vocabulary cannot be read here are not thereby safe.**
> UCE and STATE represent genes by embeddings of their protein products, which
> is how they generalise across species, so there is no flat gene list to
> intersect. That means this audit cannot bound their coverage -- not that their
> coverage is complete.

## Embed with every model that will run

One `embed_cells` call per model, each writing a row-aligned matrix to
`.obsm["X_<model>"]`, so one AnnData ends up holding several views of the same
cells.

Three rules this section follows, all of them reactions to how the previous
version of this notebook behaved:

* **No failure raises.** A missing backend, an unreachable checkpoint or an
  upstream bug goes into `FAILURES` and gets printed as a table. The notebook
  this replaces raised twice, so a reader without the optional stack got a
  traceback instead of a notebook.
* **The cache is cleared between the large transformers.** Wrappers are cached
  by `(model_key, device, kwargs)`, which is what makes chunked inference cheap,
  but eight foundation models resident at once is a way to run out of GPU
  memory rather than a speed-up.
* **Timings are recorded.** Not as a benchmark -- the hardware is whatever you
  are on -- but because a two-order-of-magnitude spread across the roster is
  itself a practical result when you are choosing what to run at scale.

In [ ]:
SWEEP: list[str] = []
FAILURES: dict[str, str] = {}
TIMINGS: dict[str, float] = {}

# Wrapper constructor kwargs, nested by model name. The nesting is not
# optional: model_kwargs is dict[str, dict[str, Any]] keyed by model, so a
# flat {"batch_key": ...} is silently ignored and the model never sees it.
MODEL_KWARGS: dict[str, dict[str, object]] = {}


def run_model(model_key, target=None, key=None, **kwargs):
    # Embed one model, timing it and recording any failure instead of
    # letting it stop the notebook. Returns True on success.
    obsm_key = key or f"X_{model_key}"
    t0 = time.perf_counter()
    try:
        embedder.embed_cells(
            target if target is not None else prepared,
            models=[model_key],
            preprocessing="none",   # part 2 already prepared the layers
            model_kwargs=MODEL_KWARGS or None,
            **kwargs,
        )
    except Exception as exc:  # noqa: BLE001 - a failed model is data, not a stop
        FAILURES[obsm_key] = f"{type(exc).__name__}: {exc}"
        TIMINGS[obsm_key] = time.perf_counter() - t0
        print(f"  {model_key:26} FAILED  {type(exc).__name__}")
        return False
    TIMINGS[obsm_key] = time.perf_counter() - t0
    SWEEP.append(obsm_key)
    print(f"  {model_key:26} ok      {TIMINGS[obsm_key]:6.1f}s")
    return True


print("classical baselines:")
for model_key in ("pca", "scvi"):
    run_model(model_key)

### The foundation models

These are the ones with a vocabulary, and the ones that cost real time. Each is
attempted independently, so one unavailable backend costs you that row and
nothing else.

In [ ]:
print("foundation models:")
for model_key in ("scgpt", "geneformer_v2_12L", "uce", "transcriptformer_sapiens",
                  "tahoe_70m"):
    run_model(model_key, batch_size=8)
    embedder.clear_model_cache()   # see the note above on resident weights

### STATE, and the constraint that comes with it

STATE is worth its own cell for two reasons that are easy to trip over.

Its `embed_cells` writes the AnnData to a temporary `.h5ad` on disk and hands
the *path* to Arc's inference code, so the object has to be h5ad-writable --
an in-memory-only view, or an `.obs` column holding an unserialisable object,
fails here and nowhere else in the roster.

And if no checkpoint is given, `load()` downloads `arcinstitute/SE-600M`, which
is roughly 12 GB. That is fine once and painful in a loop, so point it at a
local copy when you have one. The checkpoint goes in `model_kwargs` nested
under the model name, alongside anything else the wrapper constructor takes.

In [ ]:
# Point STATE at a local checkpoint when one exists, rather than triggering a
# 12 GB download. The nesting under "state" is what makes it reach the
# wrapper constructor at all.
STATE_CHECKPOINT = Path("data/checkpoints/state/SE-600M/se600m_epoch16.ckpt")
if STATE_CHECKPOINT.exists():
    MODEL_KWARGS["state"] = {"checkpoint": str(STATE_CHECKPOINT)}
    print(f"using local STATE checkpoint: {STATE_CHECKPOINT}")
else:
    print("no local STATE checkpoint; load() will download SE-600M (~12 GB)")

print("\nSTATE:")
run_model("state", batch_size=8)
embedder.clear_model_cache()
MODEL_KWARGS.pop("state", None)

### The two scvi-tools variants that exist to keep section 1 honest

`scvi` has already run above, told nothing about the data beyond the counts.
Its wrapper also accepts `batch_key`, which it forwards into
`setup_anndata` -- so it can be *told* the very covariate section 1 will score
it on removing.

Running it both ways gives the notebook its one controlled experiment: two
embeddings from the same wrapper, the same architecture and the same data,
differing in exactly one thing. Whatever gap appears between them in section 1
is the value of being told, and nothing else.

`scanvi` goes further and has no choice about it: it **raises** without
`labels_key`, so it necessarily sees the cell-type labels that
bio-conservation is scored against. That does not make it a bad model, it makes
its scores incomparable to the unsupervised ones, and section 1 marks the row
rather than averaging it in.

In [ ]:
print("integration variants:")

MODEL_KWARGS["scvi"] = {"batch_key": BATCH_KEY}
run_model("scvi", key="X_scvi_batch")
# embed_cells names the slot after the model, so rename to keep both runs.
if "X_scvi" in prepared.obsm and "X_scvi_batch" in SWEEP:
    prepared.obsm["X_scvi_batch"] = prepared.obsm.pop("X_scvi")
MODEL_KWARGS.pop("scvi", None)

# scANVI without labels_key: show the raise rather than describing it.
try:
    embedder.embed_cells(prepared, models=["scanvi"], preprocessing="none")
except Exception as exc:  # noqa: BLE001 - the point is the message
    print(f"\nscanvi with no labels_key -> {type(exc).__name__}: {exc}")

MODEL_KWARGS["scanvi"] = {"batch_key": BATCH_KEY, "labels_key": LABEL_KEY}
run_model("scanvi")
MODEL_KWARGS.pop("scanvi", None)
embedder.clear_model_cache()

## What the sweep actually produced

The failures table is a first-class result, not an appendix. A model absent from
`SWEEP` is absent from every comparison for the rest of the notebook, and each
later table names which ones those are rather than quietly showing fewer rows.

In [ ]:
EMBEDDINGS = [k for k in SWEEP if k in prepared.obsm]

shapes = pd.DataFrame(
    [{"obsm_key": k, "dim": prepared.obsm[k].shape[1],
      "seconds": round(TIMINGS.get(k, float("nan")), 1),
      "finite": bool(np.isfinite(prepared.obsm[k]).all())}
     for k in EMBEDDINGS]
).set_index("obsm_key")
display(shapes.sort_values("seconds"))

print(f"{len(EMBEDDINGS)} of {len(ROSTER) + len(INTEGRATION_VARIANTS)} attempted "
      f"models produced an embedding")
if FAILURES:
    print("\nfailures, verbatim:")
    display(pd.Series(FAILURES, name="error").to_frame())
else:
    print("no failures")

The dimensionality spread in that table is the reason [section
1](#1-do-the-models-agree) leads with a rank-based metric. The roster runs from
10 dimensions to 1280, and most similarity measures are not comparable across
widths -- a 1280-dimensional space simply has more room in which to be far
apart. Any comparison that ignores that is measuring model width as much as
model content.

One more check before comparing anything: an embedding of the right shape can
still be degenerate. A model whose vocabulary matched almost nothing returns a
near-constant matrix, which has a perfectly reasonable shape and no information
in it at all. Rank is the cheap test.

In [ ]:
rows = []
for key in EMBEDDINGS:
    M = np.asarray(prepared.obsm[key], dtype=np.float64)
    # A near-constant embedding is the signature of a vocabulary miss: the
    # shape is fine, the variance is not.
    per_dim_std = M.std(axis=0)
    rows.append({
        "obsm_key": key,
        "dim": M.shape[1],
        "effective_rank": int(np.linalg.matrix_rank(M, tol=1e-6)),
        "dead_dims": int((per_dim_std < 1e-8).sum()),
        "mean_std": per_dim_std.mean(),
    })

DEGENERACY = pd.DataFrame(rows).set_index("obsm_key")
DEGENERACY["rank_fraction"] = DEGENERACY["effective_rank"] / DEGENERACY["dim"]
display(DEGENERACY.round(4))

suspect = DEGENERACY.index[DEGENERACY["rank_fraction"] < 0.5].tolist()
print(f"embeddings using under half their dimensions: {suspect or 'none'}")

## 1. scIB: does integration preserve the biology?

scIB (Luecken et al. 2022) is an **atlas integration** benchmark. It asks one
question with two halves: when you merge batches, does the technical axis go
away, and does the biological one survive?

It is not a general embedding-quality score, and it says nothing about
perturbations -- that is [section 2](#2-cell-eval-is-the-perturbation-preserved),
on a different dataset. Nine metrics go in, and they collapse into two aggregate
columns plus scIB's 0.6/0.4 weighting of them:

| Block | Metrics | Rewards |
| --- | --- | --- |
| bio-conservation | `nmi`, `ari`, `asw_label`, `isolated_label_asw`, `clisi` | keeping cell types apart |
| batch-correction | `asw_batch`, `graph_conn`, `ilisi`, `kbet` | mixing batches together |

Those two pull against each other by construction, which is the point: the
easiest way to mix batches perfectly is to collapse every cell into one blob,
and the easiest way to preserve biology perfectly is to change nothing. `total`
is a compromise, not a truth.

`tl.compute_scib_metrics` is the only function in `tl` that takes several
`.obsm` keys at once, so the whole roster is scored in one call.

In [ ]:
t0 = time.perf_counter()
SCIB = tl.compute_scib_metrics(
    prepared,
    embedding_keys=EMBEDDINGS,
    label_key=LABEL_KEY,
    batch_key=BATCH_KEY,
)
print(f"scored {len(EMBEDDINGS)} embeddings in {time.perf_counter() - t0:.0f}s")
display(SCIB.round(3))

### Which metrics actually ran

Every scIB metric here is computed best-effort: a failure in one becomes a NaN
with a warning rather than sinking the whole comparison. That is the right
default -- the scIB suite is sensitive to installed versions, and its LISI
metrics shell out to a binary that only ships for some platforms -- but it means
a table can look complete while a column is entirely absent.

So check, rather than reading the table and assuming.

In [ ]:
BIO_COLS = ["nmi", "ari", "asw_label", "isolated_label_asw", "clisi"]
BATCH_COLS = ["asw_batch", "graph_conn", "ilisi", "kbet"]

status = []
for col in BIO_COLS + BATCH_COLS:
    if col not in SCIB.columns:
        status.append({"metric": col, "block": "bio" if col in BIO_COLS else "batch",
                       "state": "absent"})
    else:
        n_nan = int(SCIB[col].isna().sum())
        status.append({
            "metric": col,
            "block": "bio" if col in BIO_COLS else "batch",
            "state": "ok" if n_nan == 0 else f"NaN for {n_nan}/{len(SCIB)}",
        })
METRIC_STATUS = pd.DataFrame(status).set_index("metric")
display(METRIC_STATUS)

usable_bio = [c for c in BIO_COLS if c in SCIB.columns and SCIB[c].notna().any()]
usable_batch = [c for c in BATCH_COLS if c in SCIB.columns and SCIB[c].notna().any()]
print(f"bio_conservation is the mean of {len(usable_bio)}: {usable_bio}")
print(f"batch_correction is the mean of {len(usable_batch)}: {usable_batch}")

### `isolated_label_asw`, and why it exists here at all

That column is in the table above, and it would not be if this notebook had
skipped the batch covariate.

scIB counts isolated-label ASW as a **bio-conservation** metric, but it
*identifies* which labels are isolated by counting how few batches each one
appears in. So it needs the batch key even though it scores biology. Called
without one, `compute_scib_metrics` omits it entirely rather than reporting a
column of NaN.

This atlas is a good case for it. `t_cell` has seven cells across three inDrop
runs and appears in no plate-based protocol at all; `smarter` contributes only
four of the fourteen cell types. Those genuinely batch-restricted labels are the
metric's subject matter -- and they are also the ones a flat random subsample
would have deleted, which is why [part 1](#assert-the-contract-do-not-trust-it)
samples within label instead.

`04_benchmark_models.ipynb` ran with no batch key, so it scored four bio
metrics, not five. Show the difference rather than describing it.

In [ ]:
# The same embeddings, the same labels, no batch covariate. One call, purely to
# diff the column sets against the run above.
SCIB_NO_BATCH = tl.compute_scib_metrics(
    prepared, embedding_keys=EMBEDDINGS[:2], label_key=LABEL_KEY, batch_key=None
)
with_batch = set(SCIB.columns)
without = set(SCIB_NO_BATCH.columns)
print(f"columns only present with a batch key: {sorted(with_batch - without)}")
print(f"columns only present without one:      {sorted(without - with_batch) or 'none'}")
print(f"\nbio_conservation with a batch key   : "
      f"mean of {len([c for c in BIO_COLS if c in with_batch])} metrics")
print(f"bio_conservation without a batch key: "
      f"mean of {len([c for c in BIO_COLS if c in without])} metrics")
print("\nSo the two bio_conservation columns are not on the same scale, and "
      "comparing this notebook's numbers to nb04's directly would be wrong.")

### Scoring the prediction from part 1

[Part 1](#is-this-atlas-a-real-integration-benchmark) measured the covariate
before any model ran and set an expectation: with a median within-label batch
silhouette of about +0.18, the batch-correction block should genuinely separate
the models rather than saturating near 1.0 and ranking nothing.

That is a prediction, and this cell scores it rather than assuming it. The test
is the *spread* of `batch_correction`: a near-tied column ranks nothing whatever
its mean.

In [ ]:
batch_spread = float(SCIB["batch_correction"].max() - SCIB["batch_correction"].min())
bio_spread = float(SCIB["bio_conservation"].max() - SCIB["bio_conservation"].min())
DISCRIMINATES = batch_spread >= 0.05

BATCH_VERDICT = (
    f"batch_correction spans {SCIB['batch_correction'].min():.3f} to "
    f"{SCIB['batch_correction'].max():.3f} (spread {batch_spread:.3f}); "
    f"bio_conservation spans {SCIB['bio_conservation'].min():.3f} to "
    f"{SCIB['bio_conservation'].max():.3f} (spread {bio_spread:.3f}). "
    + ("Both blocks discriminate, as part 1 predicted from the within-label "
       "batch silhouette."
       if DISCRIMINATES else
       "The batch block does NOT discriminate -- it is near-tied, so it ranks "
       "nothing here regardless of its mean.")
)
print(BATCH_VERDICT)
print(f"\npart 1 predicted a {'strong' if not WEAK_BATCH else 'weak'} benchmark; "
      f"the batch block {'did' if DISCRIMINATES else 'did not'} separate the models.")
if WEAK_BATCH == DISCRIMINATES:
    print("PREDICTION MISSED -- worth understanding before trusting either number.")

### Three tiers, not one comparison

Here is the thing the table above hides, and it is the most important caveat in
this notebook.

The models were not all told the same things. `ScVIToolsWrapper.__init__` accepts
`batch_key` and forwards it into `setup_anndata`, so scVI can be *given* the very
covariate scIB scores it on removing. And scANVI **raises** without
`labels_key`, so it necessarily sees the cell-type labels that bio-conservation
is scored against.

| Tier | Models | What it saw |
| --- | --- | --- |
| told nothing | `pca`, `scgpt`, `geneformer_v2_12L`, `uce`, `transcriptformer_sapiens`, `tahoe_70m`, `state` | counts only |
| told the batch | `scvi` with `batch_key` | the covariate it is scored on removing |
| told the labels | `scanvi` | the labels bio-conservation scores |

Averaging across those tiers and calling the result a ranking would be
dishonest. `X_scvi` against `X_scvi_batch` isolates it cleanly: same wrapper,
same architecture, same data, differing in exactly one thing.

In [ ]:
PAIR = [k for k in ("X_scvi", "X_scvi_batch") if k in SCIB.index]
if len(PAIR) == 2:
    told = SCIB.loc[PAIR, ["bio_conservation", "batch_correction", "total"]]
    told.index = ["told nothing", "told the batch"]
    display(told.round(3))
    delta = (SCIB.loc["X_scvi_batch", "batch_correction"]
             - SCIB.loc["X_scvi", "batch_correction"])
    print(f"batch_correction gained by being told the covariate: {delta:+.3f}")
    print(f"bio_conservation change over the same edit: "
          f"{SCIB.loc['X_scvi_batch', 'bio_conservation'] - SCIB.loc['X_scvi', 'bio_conservation']:+.3f}")
else:
    print(f"need both X_scvi and X_scvi_batch to make this comparison; have {PAIR}")

if "X_scanvi" in SCIB.index:
    print("\nscanvi saw the labels, so its bio_conservation is not comparable "
          "to an unsupervised model's:")
    display(SCIB.loc[["X_scanvi"], ["bio_conservation", "batch_correction", "total"]].round(3))

### Where the nine metrics disagree

`bio_conservation` and `batch_correction` are means, and a mean hides
disagreement. If all five bio metrics ranked the roster identically, four of them
would be redundant. They do not, and the pairs that disagree most are worth
naming -- because a claim like "model X preserves biology best" is only as solid
as the agreement between the metrics behind it.

In [ ]:
present = [c for c in BIO_COLS + BATCH_COLS
           if c in SCIB.columns and SCIB[c].notna().sum() >= 3]
if len(present) >= 2 and len(SCIB) >= 3:
    RANK_AGREEMENT = SCIB[present].corr(method="spearman")
    display(RANK_AGREEMENT.round(2))

    off = RANK_AGREEMENT.where(~np.eye(len(RANK_AGREEMENT), dtype=bool)).stack()
    off = off.sort_values()
    print(f"least agreement: {off.index[0][0]} vs {off.index[0][1]} "
          f"(Spearman {off.iloc[0]:+.2f})")
    print(f"most agreement:  {off.index[-1][0]} vs {off.index[-1][1]} "
          f"(Spearman {off.iloc[-1]:+.2f})")
    if off.iloc[0] < 0:
        print("\nA negative correlation means those two metrics rank the roster "
              "in opposite directions. Neither is wrong; they reward different "
              "things, and the aggregate averages them anyway.")
else:
    print(f"too few models or metrics for a rank correlation "
          f"({len(SCIB)} models, {len(present)} metrics)")

### The two blocks, plotted

The only plots in this section. The numbers come from scIB, and embpy's own
purity and clustering helpers are deliberately not used -- scIB already optimises
a Leiden clustering against the label to compute NMI and ARI, so re-clustering
would recompute the same thing worse.

Read the two panels together rather than the `total` column alone. A model high
on one and low on the other has made a trade, and which trade you want depends
on what you are going to do next.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=False)
order = SCIB.sort_values("total", ascending=False).index

for ax, (block, cols, title) in zip(axes, [
    ("bio_conservation", usable_bio, "bio-conservation (keep the cell types apart)"),
    ("batch_correction", usable_batch, "batch-correction (mix the protocols)"),
]):
    SCIB.loc[order, cols].plot.bar(ax=ax, width=0.8)
    ax.plot(range(len(order)), SCIB.loc[order, block].values, "k_",
            markersize=18, markeredgewidth=2.5, label=block)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25, labelsize=8)
    ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# One orientation UMAP, on the best-scoring embedding: the biology it was asked
# to keep, and the covariate it was asked to remove, side by side.
BEST = str(SCIB.index[0])
pl.embedding_color_panel(
    prepared,
    obsm_key=BEST,
    method="umap",
    color_keys=[LABEL_KEY, BATCH_KEY],
    ncols=2,
    title=f"{BEST}: biology to keep (left), protocol to remove (right)",
)
plt.show()
print(f"{BEST} scored highest on total ({SCIB.loc[BEST, 'total']:.3f}). "
      f"A UMAP is a rendering of the embedding, not the embedding -- the numbers "
      f"above are what to conclude from.")

**What section 1 established.**

* The roster does not agree on either half of the question, and the spread in
  `BATCH_VERDICT` says by how much.
* `isolated_label_asw` is only computable because this notebook supplied a batch
  covariate, so its `bio_conservation` is a mean of five metrics where
  `04_benchmark_models.ipynb`'s was a mean of four. The two are not comparable.
* The comparison is three tiers, not one. `X_scvi` against `X_scvi_batch`
  measures what being told the covariate is worth.
* The nine metrics do not rank the roster identically, so no single number
  settles it.

None of which says whether these embeddings preserve a *perturbation*. That
needs a different experiment, and [section 2](#2-cell-eval-is-the-perturbation-preserved)
runs it on one.

## 2. cell-eval: is the perturbation preserved?

This section runs on a **different dataset**, and the reason is the mirror image
of the rule that governs [section 1](#1-scib-does-integration-preserve-the-biology).

scIB needs a technical covariate to remove and biological labels to keep, so it
needs an atlas. cell-eval scores agreement *per perturbation*, so it needs an
experiment with many perturbations -- and an atlas has none. Running cell-eval on
the atlas would produce numbers, and they would mean nothing.

[04_benchmark_models](04_benchmark_models.ipynb) shows the degenerate case
concretely. It used seven cell types as stand-in perturbations, and every
discrimination score came back exactly `1.000` for all five models:

| model | discrimination_score_l2 | discrimination_score_cosine |
| --- | --- | --- |
| pca | 1.000 | 1.0 |
| scvi | 1.000 | 1.0 |
| scgpt | 1.000 | 1.0 |
| geneformer_v2_12L | 1.000 | 1.0 |
| state | 1.000 | 1.0 |

A metric that returns the same perfect score for every model has ranked
nothing. Telling a monocyte from a T cell is trivial; the discrimination family
only becomes informative when the perturbations are numerous and some of them
are subtle.

### The dataset

**Norman et al. 2019** -- CRISPRa in K562, single and combinatorial. 111,255
cells over 19,018 genes, with 237 perturbation levels of which 236 are real
perturbations and one is the control.

Three properties earn it this section:

* **Combinatorial perturbations.** Levels are single (`KLF1`) and pairwise
  (`CEBPE+RUNX1T1`). A pair whose two members overlap in effect is genuinely
  hard to distinguish from either alone, which is what gives the discrimination
  metrics something to fail at.
* **Every level has at least 50 cells.** So nothing is dropped for size, and a
  `DROPPED` list that comes back empty below is a real result rather than a
  filter that silently did nothing.
* **A named control.** The column carries a literal `"control"` level with
  11,835 cells, which cell-eval needs as its baseline.

One trap, and it is the same one the atlas loader guards against: **`.X` here is
log-normalised, not counts** (`X.max()` is 8.5, non-integral). The raw counts are
in `layers["counts"]`. Two of the four models in this section declare
`input_layer="counts"`, so taking `.X` at face value would fit a
negative-binomial likelihood to non-integers and never complain.

In [ ]:
PERT_KEY = "perturbation_name"     # 237 levels; see the note on the alternatives
CONTROL_VALUE = "control"          # 11,835 cells
N_PERT_CELLS = 4000                # subsample; the full set is 111,255 cells
MIN_CELLS_PER_PERT = 4             # cell-eval needs each level on both sides
PERT_PATH = DATA_DIR / "norman2019.h5ad"


def load_norman():
    # Two other columns look like candidates and both are unusable:
    # `guide_ids` has the same 237 levels but labels the control as the empty
    # string, and `guide_identity` has 290 levels of raw guide pairs
    # (NegCtrl10_NegCtrl0__NegCtrl10_NegCtrl0) with no clean control level.
    if PERT_PATH.exists():
        return sc.read_h5ad(PERT_PATH)
    import pertpy as pt  # noqa: PLC0415 - fallback only

    fetched = pt.data.norman_2019()
    fetched.write_h5ad(PERT_PATH)
    return fetched


pert_adata = load_norman()
print(f"loaded {pert_adata.n_obs} cells x {pert_adata.n_vars} genes")

# .X is log-normalised; the counts are in a layer. Take the counts, or the two
# scvi-tools models silently fit a count likelihood to non-integers.
assert COUNTS_LAYER in pert_adata.layers, f"expected raw counts in .layers[{COUNTS_LAYER!r}]"
pert_adata.X = pert_adata.layers[COUNTS_LAYER].copy()

# Drop the paper's own embeddings. Carrying them into a notebook that scores
# embeddings invites mistaking one for ours.
for key in list(pert_adata.obsm):
    del pert_adata.obsm[key]

X = pert_adata.X
print(f"counts integral after switching layers: "
      f"{float(np.abs(X.data - np.round(X.data)).max()) == 0.0}")
print(f"{PERT_KEY}: {pert_adata.obs[PERT_KEY].nunique()} levels")
print(f"control cells: {int((pert_adata.obs[PERT_KEY] == CONTROL_VALUE).sum())}")

Subsampling has to happen **within** perturbation, not across the whole object.
A flat random subsample of 4,000 from 111,255 cells would drop most of the 237
levels entirely and leave the rest with a handful of cells each -- which would
turn the level count, the thing that makes this dataset worth using, into the
thing the subsample destroyed.

In [ ]:
rng = np.random.default_rng(SEED)

# Take an equal quota per level rather than a proportional sample, so the rare
# perturbations survive and the abundant ones stop dominating the averages.
levels = pert_adata.obs[PERT_KEY].astype(str)
per_level = max(MIN_CELLS_PER_PERT, N_PERT_CELLS // levels.nunique())

keep = []
for level in levels.unique():
    idx = np.flatnonzero((levels == level).to_numpy())
    if len(idx) > per_level:
        idx = rng.choice(idx, per_level, replace=False)
    keep.append(idx)
keep = np.sort(np.concatenate(keep))

pert_adata = pert_adata[keep].copy()
pert_adata.obs[PERT_KEY] = pert_adata.obs[PERT_KEY].astype(str).astype("category")
pert_adata.layers[COUNTS_LAYER] = pert_adata.X.copy()

counts_per_level = pert_adata.obs[PERT_KEY].value_counts()
print(f"{pert_adata.n_obs} cells, {counts_per_level.size} levels, "
      f"{per_level} cells per level target")
print(f"cells per level: min {counts_per_level.min()}, "
      f"median {int(counts_per_level.median())}, max {counts_per_level.max()}")
display(counts_per_level.head(8).to_frame("n_cells"))

### A reduced roster, and why

Section 1 swept eight models. This section runs **four** -- `pca`, `scvi`,
`geneformer_v2_12L` and `state` -- and the reason is worth stating rather than
leaving a reader to wonder where the other four went.

The question here is what the *metric family* measures, not which of eight
models wins. Four models are enough to show a spread, and these four are the
ones already proven end to end, spanning a classical baseline, a VAE and two
foundation models. Embedding 237 perturbation levels eight times to make the
same point would cost hours and add nothing.

In [ ]:
PERT_ROSTER = ["pca", "scvi", "geneformer_v2_12L", "state"]

PERT_SWEEP: list[str] = []
PERT_FAILURES: dict[str, str] = {}

prepared_pert = pp.preprocess_counts(
    pert_adata, pipeline="standard", min_genes=0, min_cells=0,
    target_sum=1e4, log_transform=True, n_top_genes=2000, select_hvg=True,
    scale=False, copy=True,
)

for model_key in PERT_ROSTER:
    t0 = time.perf_counter()
    try:
        embedder.embed_cells(
            prepared_pert, models=[model_key], preprocessing="none", batch_size=8
        )
    except Exception as exc:  # noqa: BLE001 - a failed model is data, not a stop
        PERT_FAILURES[f"X_{model_key}"] = f"{type(exc).__name__}: {exc}"
        print(f"  {model_key:20} FAILED  {type(exc).__name__}")
    else:
        PERT_SWEEP.append(f"X_{model_key}")
        print(f"  {model_key:20} ok      {time.perf_counter() - t0:6.1f}s")
    embedder.clear_model_cache()

PERT_EMBEDDINGS = [k for k in PERT_SWEEP if k in prepared_pert.obsm]
print(f"\n{len(PERT_EMBEDDINGS)} of {len(PERT_ROSTER)} models embedded")
if PERT_FAILURES:
    display(pd.Series(PERT_FAILURES, name="error").to_frame())

### Building a pair cell-eval will accept

cell-eval compares a **(predicted, real)** pair of AnnData objects. This notebook
has no prediction model, so the pair is two halves of the same dataset -- which
measures an *agreement ceiling*, not prediction accuracy. Say that plainly,
because the numbers look like accuracy and are not.

The construction has one hard requirement that is easy to violate: both objects
must carry the **same set of perturbation levels**, or cell-eval raises
`Perturbation mismatch`. A plain random half-split fails that as soon as any
level is small enough to land entirely on one side. So the split runs *within*
each level, and levels too small to appear on both sides are dropped with the
count shown.

In [ ]:
levels = prepared_pert.obs[PERT_KEY].astype(str)
level_counts = levels.value_counts()
USABLE = level_counts[level_counts >= MIN_CELLS_PER_PERT].index.tolist()
DROPPED = sorted(set(level_counts.index) - set(USABLE))

paired = prepared_pert[levels.isin(USABLE).to_numpy()].copy()
is_pred = np.zeros(paired.n_obs, dtype=bool)
for level in USABLE:
    idx = np.flatnonzero((paired.obs[PERT_KEY].astype(str) == level).to_numpy())
    rng.shuffle(idx)
    is_pred[idx[: len(idx) // 2]] = True

real = paired[~is_pred].copy()
pred = paired[is_pred].copy()

print(f"real {real.n_obs} cells | pred {pred.n_obs} cells")
print(f"levels on both sides: {len(USABLE)}")
print(f"dropped (fewer than {MIN_CELLS_PER_PERT} cells): {DROPPED or 'none'}")
assert set(real.obs[PERT_KEY].astype(str)) == set(pred.obs[PERT_KEY].astype(str)), \
    "cell-eval requires identical level sets on both sides"
assert CONTROL_VALUE in set(real.obs[PERT_KEY].astype(str)), "control level missing"
print("level sets match")

### Pointing every metric at the embedding

This is the part that silently goes wrong, and it went wrong in
[04_benchmark_models](04_benchmark_models.ipynb) before it was caught.

cell-eval's metrics fall into families. The `MetricType.ANNDATA_PAIR` ones can
score *either* the expression matrix or an `.obsm` embedding, and which one they
use is decided per metric by an `embed_key` entry in `metric_configs`. A metric
you forget to configure does not error -- it scores `.X` instead, and its number
lands in the same table beside the ones that scored your embedding.

nb04 configured 4 of 10 pair metrics. Once the remaining six were pointed at the
embedding, `pearson_delta` moved from 0.777 to 0.854 -- so the original table was
a mixture of two different measurements presented as one.

The fix is to stop hand-listing metric names and **enumerate the registry**, then
print the count so a reader can see none were missed.

One metric resists this. `discrimination_score_l1` hardcodes `embed_key = None`
upstream, so it cannot be made to read `.obsm` no matter what you pass. It is
skipped explicitly rather than left in: a single expression-space column sitting
in an embedding-space table is precisely the mixed result this whole subsection
is about avoiding.

In [ ]:
from cell_eval import MetricType, metrics_registry

# Enumerate rather than hand-list: a metric added by a future cell-eval release
# is picked up automatically instead of silently scoring .X.
ALL_PAIR = set(metrics_registry.list_metrics(MetricType.ANNDATA_PAIR))

# discrimination_score_l1 hardcodes embed_key = None upstream, so it always
# scores .X. Skip it rather than let one expression-space column sit in an
# embedding-space table.
UNCONFIGURABLE = {"discrimination_score_l1"}
PAIR_METRICS = sorted(ALL_PAIR - UNCONFIGURABLE)

print(f"ANNDATA_PAIR metrics: {len(ALL_PAIR)}")
print(f"configurable on an embedding: {len(PAIR_METRICS)} -> {PAIR_METRICS}")
print(f"skipped (ignores embed_key): {sorted(UNCONFIGURABLE)}")

In [ ]:
frames = {}
for key in PERT_EMBEDDINGS:
    # Every pair metric gets the same embed_key. Configuring some and not
    # others is what produced nb04's mixed table.
    metric_configs = {name: {"embed_key": key} for name in PAIR_METRICS}
    real.obsm[key] = prepared_pert.obsm[key][~is_pred]
    pred.obsm[key] = prepared_pert.obsm[key][is_pred]

    t0 = time.perf_counter()
    try:
        per_pert, agg = tl.cell_eval(
            adata_pred=pred,
            adata_real=real,
            control_pert=CONTROL_VALUE,
            pert_col=PERT_KEY,
            profile="anndata",       # runs every ANNDATA_PAIR metric, which is
                                     # exactly the family that can see an .obsm
            skip_de=True,            # DE metrics score .X by construction; this
                                     # goes to the evaluator *constructor*
            metric_configs=metric_configs,
            skip_metrics=sorted(UNCONFIGURABLE),
        )
    except Exception as exc:  # noqa: BLE001
        print(f"  {key:22} FAILED  {type(exc).__name__}: {exc}")
        continue
    frames[key] = per_pert
    print(f"  {key:22} ok  {len(per_pert)} rows  {time.perf_counter() - t0:5.1f}s")

print(f"\nconfigured {len(PAIR_METRICS)} pair metrics per embedding, "
      f"{len(frames)} embeddings scored")

`tl.cell_eval` returns a **tuple** of two frames -- per-perturbation and
aggregate -- not one. That matters because the pipeline-friendly variant behaves
differently in a way its own docstring gets wrong, which is worth one short
subsection.

### What `run_cell_eval` does differently

`tl.run_cell_eval` is the pipeline form of the same call. Its summary line says
it returns *"a single DataFrame combining per-perturbation and aggregate
results"*. It does not. The body returns only the per-perturbation frame and
writes the aggregate into `adata_real.uns["cell_eval_agg"]` -- **mutating the
object you passed in as `real`**.

Neither half of that is a problem once you know it. Both halves are a problem if
you trust the summary line: you lose the aggregate, and an object you thought was
read-only has grown a key.

In [ ]:
if PERT_EMBEDDINGS:
    probe_key = PERT_EMBEDDINGS[0]
    real_probe = real.copy()
    before = set(real_probe.uns)
    tidy = tl.run_cell_eval(
        adata_pred=pred, adata_real=real_probe,
        control_pert=CONTROL_VALUE, pert_col=PERT_KEY, profile="anndata",
        skip_de=True,
        metric_configs={n: {"embed_key": probe_key} for n in PAIR_METRICS},
        skip_metrics=sorted(UNCONFIGURABLE),
    )
    print(f"run_cell_eval returned: {type(tidy).__name__} with {len(tidy)} rows")
    print(f"keys added to the object passed as `real`: "
          f"{sorted(set(real_probe.uns) - before)}")
    del real_probe

### The result

One row per embedding, averaged over perturbations. The columns present depend
on the installed cell-eval version, so this takes whatever is numeric rather
than assuming a fixed set -- a hand-written column list is how a notebook breaks
on a dependency bump.

In [ ]:
if frames:
    summary = {}
    for key, frame in frames.items():
        numeric = frame.select_dtypes(include=[np.number])
        summary[key] = numeric.mean()
    CELL_EVAL = pd.DataFrame(summary).T
    CELL_EVAL.index.name = "embedding"
    display(CELL_EVAL.round(3))
else:
    CELL_EVAL = pd.DataFrame()
    print("no cell-eval results to show")

Compare the discrimination columns against nb04's, reprinted at the top of this
section. There, with seven well-separated stand-in perturbations, every model
scored exactly 1.000. Here, with 236 real perturbations including combinatorial
ones, the same metric has room to separate them -- and if it still does not, that
is now a finding about the models rather than an artefact of the task being
trivial.

In [ ]:
if not CELL_EVAL.empty:
    # Discrimination scores and error metrics point in opposite directions, so
    # they get separate panels rather than a shared axis that would flatter one.
    higher_better = [c for c in CELL_EVAL.columns if "discrimination" in c
                     or c.startswith("pearson")]
    lower_better = [c for c in CELL_EVAL.columns
                    if c.startswith(("mse", "mae")) or "edistance" in c]

    panels = [(higher_better, "higher is better"), (lower_better, "lower is better")]
    panels = [(cols, title) for cols, title in panels if cols]
    fig, axes = plt.subplots(1, len(panels), figsize=(7 * len(panels), 3.6))
    axes = np.atleast_1d(axes)
    for ax, (cols, title) in zip(axes, panels):
        CELL_EVAL[cols].plot.bar(ax=ax, width=0.8)
        ax.set_title(title)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=20)
        ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()

### Does it find the strong perturbations?

An aggregate over 236 perturbations can hide the thing that matters most: does
the metric respond to how large the perturbation actually is? A score that rates
a near-null perturbation as highly as a strong one is not measuring the
perturbation -- it is measuring something else and averaging it.

So rank the perturbations by a measured effect size, computed independently of
cell-eval, and check whether the per-perturbation scores track it.

In [ ]:
if frames:
    # Effect size computed independently of cell-eval: the L2 distance between
    # each perturbation's mean expression profile and the control's.
    ctrl_mask = (prepared_pert.obs[PERT_KEY].astype(str) == CONTROL_VALUE).to_numpy()
    log_layer = prepared_pert.layers.get("log_normalized", prepared_pert.X)
    log_layer = np.asarray(
        log_layer.todense() if hasattr(log_layer, "todense") else log_layer
    )
    ctrl_mean = log_layer[ctrl_mask].mean(axis=0)

    effect = {}
    for level in USABLE:
        if level == CONTROL_VALUE:
            continue
        m = (prepared_pert.obs[PERT_KEY].astype(str) == level).to_numpy()
        effect[level] = float(np.linalg.norm(log_layer[m].mean(axis=0) - ctrl_mean))
    EFFECT = pd.Series(effect, name="effect_l2").sort_values(ascending=False)

    print(f"effect size across {len(EFFECT)} perturbations: "
          f"{EFFECT.min():.2f} to {EFFECT.max():.2f}")
    display(pd.concat([EFFECT.head(5), EFFECT.tail(5)]).to_frame().round(3))

In [ ]:
if frames:
    # Correlate each metric against the independent effect size, per embedding.
    rows = []
    for key, frame in frames.items():
        idx_col = frame.columns[0] if frame.index.name is None else None
        f = frame.copy()
        if idx_col is not None and f[idx_col].dtype == object:
            f = f.set_index(idx_col)
        common = [i for i in f.index.astype(str) if i in EFFECT.index]
        if len(common) < 5:
            continue
        aligned = EFFECT.loc[common]
        for col in f.select_dtypes(include=[np.number]).columns:
            rows.append({
                "embedding": key,
                "metric": col,
                "spearman_vs_effect": float(
                    pd.Series(f.loc[common, col].values, index=common)
                    .corr(aligned, method="spearman")
                ),
            })

    if rows:
        TRACKS_EFFECT = (
            pd.DataFrame(rows)
            .pivot(index="metric", columns="embedding", values="spearman_vs_effect")
        )
        display(TRACKS_EFFECT.round(3))
        print("A metric near zero here does not respond to perturbation strength.")
    else:
        print("not enough shared perturbation levels to correlate")

### Regression is a third question, and a third function

Two entry points have now been used, and there is a third that answers something
different again. `tl.benchmark_embeddings` trains ordinary regressors --
linear, ridge, k-NN, random forest -- on the embedding and reports `mse`, `r2`,
`pearson`, `spearman`.

It contains **no scIB path** and no cell-eval path: `grep -in scib
src/embpy/tl/benchmark.py` returns nothing. So the package offers three separate
front doors and no single "benchmark this embedding" call, which is worth
knowing before you go looking for one.

| Function | Question | Needs |
| --- | --- | --- |
| `tl.compute_scib_metrics` | does integration keep the biology? | labels + a batch covariate |
| `tl.cell_eval` | do two objects agree per perturbation? | a (pred, real) pair |
| `tl.benchmark_embeddings` | how much signal about a target does this carry? | a prediction target |

## Reading a cell model's attention

Every number so far came from a pooled vector: one row per cell, the model's
whole opinion compressed. Attention is the layer underneath -- which *genes* the
model looked at while forming that opinion.

For a cell model this is unusually interpretable, because the tokens are genes.
An attention row is a distribution over the genes in one cell, so asking "did
this cell's attention concentrate on its own marker genes?" is a question with a
checkable answer.

Two models in the roster can answer it, for different reasons:

* **`geneformer_v2_12L`** is a HuggingFace BERT underneath, so
  `extract_attention` takes the native `output_attentions` path and returns
  `(batch, heads, seq, seq)`.
* **`tahoe_70m`** is not HuggingFace, so it goes through a forward pre-hook that
  flips `need_weights` back on. It is the one single-cell wrapper whose capture
  is *verified* on this cluster rather than inferred -- `(4, 8, 1606, 1606)` from
  a four-cell embed.

The indexing helpers matter here and are easy to get backwards.
`tl.block_to_attention_index(b) == b`, but
`tl.block_to_hidden_state_index(b) == b + 1`, because hidden-state index 0 is the
embedding layer before any transformer block has run.

In [ ]:
ATTENTION_MODEL = "geneformer_v2_12L"

# A small slice: attention is (batch, heads, seq, seq), and seq is the number of
# ranked gene tokens, so the tensor grows quadratically in a hurry.
N_ATTN_CELLS = 4
attn_slice = prepared[:N_ATTN_CELLS].copy()

ATTENTION = None
if f"X_{ATTENTION_MODEL}" in EMBEDDINGS:
    wrapper = embedder._get_or_load_singlecell_wrapper(
        ATTENTION_MODEL, batch_size=N_ATTN_CELLS, device_str="auto"
    )
    try:
        ATTENTION = wrapper.extract_attention(attn_slice, layers=None)
    except Exception as exc:  # noqa: BLE001 - report, do not stop
        print(f"extract_attention failed: {type(exc).__name__}: {exc}")
    else:
        blocks = sorted(ATTENTION)
        first = ATTENTION[blocks[0]]
        print(f"{ATTENTION_MODEL}: {len(blocks)} blocks captured, keys {blocks}")
        print(f"per-block shape: {tuple(first.shape)}  "
              f"(batch, heads, seq, seq)")
        print(f"block -> attention index: {tl.block_to_attention_index(0)}")
        print(f"block -> hidden-state index: {tl.block_to_hidden_state_index(0)}"
              f"  (0 is the embedding layer)")
else:
    print(f"{ATTENTION_MODEL} is not in EMBEDDINGS; attention section skipped")

### What the heads are doing

Raw attention tensors are too large to read. `tl` provides summaries that
collapse them into something inspectable, and the two most useful ask opposite
questions: how *spread out* is each head's attention, and how much attention does
each gene *receive*.

A head with near-maximal entropy is attending to everything equally, which is
another way of saying it is attending to nothing. A head with low entropy has
picked a few genes. Both exist in a trained model, and the mix per layer is
informative.

In [ ]:
if ATTENTION:
    blocks = sorted(ATTENTION)
    rows = []
    for block in blocks:
        entropy = tl.attention_entropy(ATTENTION[block])
        uniformity = tl.head_uniformity(ATTENTION[block])
        rows.append({
            "block": block,
            "mean_entropy": float(np.mean(entropy)),
            "min_entropy": float(np.min(entropy)),
            "head_uniformity": float(np.mean(uniformity)),
        })
    ATTN_SUMMARY = pd.DataFrame(rows).set_index("block")
    display(ATTN_SUMMARY.round(3))

    ranked = tl.rank_layers(ATTENTION)
    print(f"layers ranked by informativeness: {ranked}")

### Does attention land on the marker genes?

The sharper test. Derive a marker set from the data -- not a hand-picked
signature, which would not transfer when the atlas is swapped -- and ask whether
cells of that label concentrate attention on it, using another label as the
contrast.

Two honest caveats before the number appears. This is four cells, so it is an
illustration and not a measurement. And **attention is not attribution**: a head
attending to a gene does not establish that the gene drove the embedding. It
says where the model looked, which is a weaker and different claim.

In [ ]:
if ATTENTION:
    # Markers from the data, so this survives a change of dataset.
    marker_source = prepared.copy()
    sc.tl.rank_genes_groups(
        marker_source, groupby=LABEL_KEY, method="wilcoxon", n_genes=25
    )
    label_of_interest = str(prepared.obs[LABEL_KEY].value_counts().index[0])
    MARKERS = [
        str(g) for g in
        marker_source.uns["rank_genes_groups"]["names"][label_of_interest]
    ]
    print(f"markers for {label_of_interest!r}: {MARKERS[:8]} ...")

    block = sorted(ATTENTION)[-1]     # the last block, closest to the output
    to_set = tl.attention_to_gene_set(
        ATTENTION[block], gene_names=list(attn_slice.var_names), gene_set=MARKERS
    )
    received = tl.received_attention(ATTENTION[block])
    print(f"\nattention mass on the marker set, per cell: "
          f"{np.round(np.asarray(to_set).ravel(), 4).tolist()}")
    print(f"labels of those cells: "
          f"{attn_slice.obs[LABEL_KEY].astype(str).tolist()}")
    print(f"received-attention vector length: {np.asarray(received).size}")
    del marker_source

### Where attention is not available, and why

Two models in the roster return `has_attention = False`, and both are honest
refusals with a specific cause rather than an unimplemented feature:

* **`scgpt`** builds `FlashMHA` unconditionally (`singlecell_models.py:724`).
  Flash attention fuses the softmax into the kernel and never materialises the
  weight matrix, so there is nothing to capture -- the weights do not exist at
  any point.
* **`state`** uses a fused `F.scaled_dot_product_attention`
  (`singlecell_models.py:980`), for the same reason.

That is a real architectural trade-off rather than an embpy limitation: the
kernels that make these models fast are the kernels that discard the weights.

### Two flags that promise what they cannot deliver

Keeping the catalogue honest cuts both ways, so here is where embpy's own
metadata is wrong.

`SingleCellWrapper` sets `has_attention = True` as its class default
(`singlecell_models.py:504`), and only `scgpt` and `state` override it. So
**`PCAEmbedding` and `ScVIToolsWrapper` both advertise attention**, and neither
can deliver it -- there is no attention in a PCA or a VAE to begin with.

The two fail differently, and the difference matters:

* **`PCAEmbedding`** raises `NotImplementedError`, which is the right answer
  reached for the wrong reason. `torch_module()` reads `self._model`, and PCA
  keeps its fit state in `self._pca`, so the module lookup returns `None`. The
  outcome is correct; only the advertised flag is wrong.
* **`ScVIToolsWrapper`** raises the same error while a real `torch.nn.Module`
  sits one attribute away. `torch_module()` reads `self._model`, but
  `embed_cells` assigns the trained model to `self._trained_model`
  (`singlecell_models.py:1856`). The module is reachable at
  `wrapper._trained_model.module` -- the same object `decode_cells` uses from
  line 1913 onward. So the introspection API cannot see a module that
  demonstrably exists.

`StackWrapper` inherits `True` with no `_get_layer_modules` override and no
recorded evidence either way, which makes it an unverified promise rather than a
known failure.

Show the raise, then show the module, because a claim like this is only worth
making if it is demonstrated.

In [ ]:
rows = []
for model_key in ("pca", "scvi"):
    if f"X_{model_key}" not in EMBEDDINGS:
        continue
    w = embedder._get_or_load_singlecell_wrapper(model_key, batch_size=8,
                                                 device_str="auto")
    try:
        w.extract_attention(prepared[:2].copy(), layers=None)
        outcome = "returned something"
    except NotImplementedError as exc:
        outcome = f"NotImplementedError: {str(exc)[:60]}"
    except Exception as exc:  # noqa: BLE001
        outcome = f"{type(exc).__name__}: {str(exc)[:60]}"

    # Is a torch module actually reachable, by any route?
    via_api = w.torch_module() is not None
    via_attr = getattr(getattr(w, "_trained_model", None), "module", None) is not None
    rows.append({
        "model": model_key,
        "has_attention": w.has_attention,
        "torch_module() finds one": via_api,
        "one exists at _trained_model.module": via_attr,
        "extract_attention": outcome,
    })

if rows:
    display(pd.DataFrame(rows).set_index("model"))
    print("has_attention=True with no reachable module is a metadata bug, "
          "not a missing feature.")

## Decode: from latent back to expression

Most of the roster is an encoder only. Five registry keys declare
`supports_decode` -- `pca`, `scvi`, `scanvi`, `totalvi` and `state` -- and for
those the latent space can be projected back to gene expression.

That makes a check available which no similarity metric provides: **how much of
the original cell survives the round trip?** An embedding that reconstructs
expression well has kept the information; one that reconstructs badly has thrown
some away, whatever its scIB score says.

`stack` is the only `supports_generation` entry in the registry -- it can
synthesise whole profiles rather than merely reconstruct them. It fails inside
arc-stack's own h5ad reader on this environment, which is why this notebook has
no generation section. That is a real gap, not an omission.

In [ ]:
DECODERS = [k for k in ("pca", "scvi", "state")
            if f"X_{k}" in EMBEDDINGS and singlecell_info(k).supports_decode]
print(f"decode-capable models present: {DECODERS}")

rows = []
for model_key in DECODERS:
    obsm_key = f"X_{model_key}"
    try:
        decoded = embedder.decode_cells(
            prepared, model=model_key, obsm_key=obsm_key,
            write_layer=f"{obsm_key}_reconstructed",
        )
    except Exception as exc:  # noqa: BLE001
        print(f"  {model_key:8} decode failed: {type(exc).__name__}: {exc}")
        continue

    # Compare against the log-normalised layer, which is what the decoders
    # target -- not the raw counts.
    target = prepared.layers.get("log_normalized", prepared.X)
    target = np.asarray(target.todense() if hasattr(target, "todense") else target)
    recon = np.asarray(decoded.todense() if hasattr(decoded, "todense") else decoded)

    # Per-gene correlation, then the median across genes: a single global
    # correlation is dominated by the mean expression profile and looks
    # flatteringly high for any method.
    per_gene = []
    for j in range(0, target.shape[1], max(1, target.shape[1] // 500)):
        a, b = target[:, j], recon[:, j]
        if a.std() > 1e-8 and b.std() > 1e-8:
            per_gene.append(float(np.corrcoef(a, b)[0, 1]))
    rows.append({
        "model": model_key,
        "dim": prepared.obsm[obsm_key].shape[1],
        "median_gene_r": float(np.median(per_gene)) if per_gene else np.nan,
        "genes_sampled": len(per_gene),
    })

if rows:
    RECONSTRUCTION = pd.DataFrame(rows).set_index("model")
    display(RECONSTRUCTION.round(3))
    print("Median per-gene correlation. A global correlation would be dominated "
          "by the mean expression profile and flatter every method.")

## Annotating the same object

The embeddings, the scIB scores and the reconstructions all live on one AnnData,
and the annotation layer attaches to the same object. Two sides to it, and only
one applies cleanly to a cell-by-gene matrix.

**The `.var` side works.** Rows are cells but columns are genes, so gene-level
annotation lands in `.var` and describes the features every model consumed.

**The `.obs` side mostly does not.** `tl.annotate_gene_perturbations` and
`tl.annotate_drug_perturbations` expect one row per *perturbation*, not one row
per cell. On this atlas the rows are cells, so there is nothing for them to key
on. [05_annotate_entities](05_annotate_entities.ipynb) covers that layer properly
and this notebook does not duplicate it.

> **embpy has no cell-type annotator.** No marker-based classifier, no
> reference-mapping method. Every `cell_type` label in this notebook arrived with
> the dataset, and scIB scores the embeddings *against* those labels. If your
> data has no labels, section 1 cannot run at all -- which is a genuine
> limitation of the metric, not something a better embedding would fix.

In [ ]:
# Gene-level annotation goes to .var and describes the features, not the cells.
# Take the top HVGs rather than all 19k: the annotators are network-bound and
# the point is the shape of the result, not a full sweep.
N_GENES_TO_ANNOTATE = 12
top_genes = (
    prepared.var.sort_values("dispersions_norm", ascending=False).index[:N_GENES_TO_ANNOTATE]
    if "dispersions_norm" in prepared.var.columns
    else prepared.var_names[:N_GENES_TO_ANNOTATE]
)
print(f"annotating {len(top_genes)} genes: {list(top_genes[:6])} ...")

gene_frame = ad.AnnData(
    X=np.zeros((len(top_genes), 1), dtype=np.float32),
    obs=pd.DataFrame({"symbol": list(top_genes)}, index=list(top_genes)),
)
try:
    gene_frame = tl.annotate_gene_perturbations(
        gene_frame, column="symbol", sources=["pathways"]
    )
except Exception as exc:  # noqa: BLE001 - live APIs, degrade to a report
    print(f"gene annotation unavailable: {type(exc).__name__}: {exc}")
else:
    cols = [c for c in gene_frame.obs.columns if c.startswith("gene_")]
    display(gene_frame.obs[cols].head(8))

    # Carry the result back onto .var, where it describes the features.
    for col in cols:
        prepared.var[col] = gene_frame.obs[col].reindex(prepared.var_names)
    print(f"columns added to .var: {cols}")

Note the shape of that call: a list of gene symbols had to be wrapped in an
AnnData with a placeholder `np.zeros` matrix. The `tl.annotate_*` family is
AnnData-in, AnnData-out -- it joins on an identifier column and writes prefixed
columns into `.obs`, and never reads `.X` at all. There is no list-in,
frame-out form, so the zero matrix is boilerplate rather than data.

## Save the artifact

An artifact you cannot read back is not an artifact. Write it, reload it, and
check that the embeddings *and* the provenance survived -- `.obsm` matrices are
the easy part, and `.uns` is where round trips usually break, because h5ad has
to serialise a nested dictionary of mixed types.

In [ ]:
ARTIFACT = OUTPUT_DIR / "cell_embeddings.h5ad"
prepared.write_h5ad(ARTIFACT)
print(f"wrote {ARTIFACT} ({ARTIFACT.stat().st_size / 1e6:.1f} MB)")

reloaded = sc.read_h5ad(ARTIFACT)
obsm_survived = [k for k in EMBEDDINGS if k in reloaded.obsm]
print(f"\n.obsm keys recovered: {len(obsm_survived)} of {len(EMBEDDINGS)}")
missing = sorted(set(EMBEDDINGS) - set(obsm_survived))
if missing:
    print(f"LOST: {missing}")

# Values, not just keys: a key that survives with mangled values is worse than
# one that is missing, because nothing downstream will notice.
identical = all(
    np.allclose(np.asarray(prepared.obsm[k]), np.asarray(reloaded.obsm[k]))
    for k in obsm_survived
)
print(f"matrices identical after the round trip: {identical}")

prov = reloaded.uns.get("embpy_cell_embeddings")
if prov is None:
    print("provenance LOST: .uns['embpy_cell_embeddings'] did not survive")
else:
    print(f"provenance keys: {sorted(prov)[:8]}")
    if "__preprocessing__" in prov:
        display(pd.Series(dict(prov["__preprocessing__"])).to_frame("preprocessing"))

## What we found

**The measured verdict**, reprinted so the conclusion sits beside the evidence
rather than being asserted from memory:

In [ ]:
print(BATCH_VERDICT if "BATCH_VERDICT" in globals() else "section 1 did not run")
print()
if "SCIB" in globals() and not SCIB.empty:
    print("scIB, best by total:")
    display(SCIB[["bio_conservation", "batch_correction", "total"]].round(3))
if "CELL_EVAL" in globals() and not CELL_EVAL.empty:
    print("cell-eval, averaged over perturbations:")
    display(CELL_EVAL.round(3))

**Six things worth carrying away.**

1. **The two metric families need different experiments.** scIB is an atlas
   integration benchmark and cell-eval is a perturbation-prediction benchmark.
   Neither substitutes for the other, and running either on the wrong dataset
   produces numbers rather than answers -- which is why this notebook loads two
   objects instead of one.
2. **Check the covariate before you score it.** A batch variable with a
   near-zero silhouette gives every model a high `batch_correction` and ranks
   nothing. Part 1 measures it up front so section 1's column can be read for
   what it is.
3. **A high batch score is only good if the covariate was a nuisance.** Rewarding
   a model for erasing a difference you care about is the same arithmetic as
   rewarding it for removing one you do not.
4. **The comparison is not apples to apples unless you say what each model was
   told.** `scvi` can be handed the batch key; `scanvi` cannot run without the
   labels. `X_scvi` against `X_scvi_batch` measures the difference that makes.
5. **Vocabulary overlap bounds everything downstream.** A `symbol`-vocabulary
   model given Ensembl IDs returns empty embeddings with no error, and for a
   rank tokeniser a partial overlap changes which genes make the cut -- so it
   alters the representation of the genes that *were* recognised.
6. **Point every pair metric at the embedding, or check which did not get
   pointed.** An unconfigured cell-eval metric scores `.X` and sits in the same
   table looking identical. One of them cannot be configured at all.

**Where to go next.**

* [04_benchmark_models](04_benchmark_models.ipynb) -- the short version of both
  sections, five models on one dataset.
* [05_annotate_entities](05_annotate_entities.ipynb) -- the annotation layer, on
  entities where it applies row-wise.
* [genes](genes.ipynb), [proteins](proteins.ipynb),
  [small molecules](small_molecules.ipynb) -- the other modality deep dives.
  Those are where embpy's own comparison metrics do the work, because no
  community standard exists for them.